# Q2. (d) Parameter Estimation in Logistic Regression

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

In [ ]:
# INITIAL VARIABLES & CONFIGURATION

# TODO: Set the true weight vector w* = [1.5, -2.0, 0.5]
TRUE_WEIGHTS = np.array([1.5, -2.0, 0.5])

# TODO: Define the sample sizes N = [15, 25, 50, 100, 200, 400, 800, 1600, 3200, 6400, 12800]
SAMPLE_SIZES = [15, 25, 50, 100, 200, 400, 800, 1600, 3200, 6400, 12800]

# Optional random seed for data generation reproducibility
DEFAULT_SEED = 2024514

In [ ]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    """
    Computes the sigmoid function: Ïƒ(z) = 1 / (1 + e^(-z))
    """
    # TODO: Implement the sigmoid function
    return 1.0 / (1.0 + np.exp(-z))


In [ ]:
def generate_synthetic_data(N: int, w_true: np.ndarray = TRUE_WEIGHTS, seed: int = DEFAULT_SEED) -> tuple[np.ndarray, np.ndarray]:
    """
    Generates a synthetic binary classification dataset.
    """
    np.random.seed(seed)

    # TODO Step 1: Determine d (number of features) from w_true
    d = len(w_true)
    # TODO Step 2: Draw feature matrix X ~ Standard Normal (N x d)
    X = np.random.randn(N, d)
    # TODO Step 3: Compute logits = X @ w_true
    logits = X @ w_true
    # TODO Step 4: Compute probabilities = sigmoid(logits)
    probabilities = sigmoid(logits)
    # TODO Step 5: Sample target array y ~ Bernoulli(probabilities) using np.random.binomial
    y = np.random.binomial(1, probabilities)

    return X, y


In [ ]:
def fit_models(X: np.ndarray, y: np.ndarray, C_map: float = 1.0) -> tuple[float, float]:
    """
    Fits unregularized MLE and MAP (L2-regularized) Logistic Regression models.
    """
    # TODO Step 1: Fit Unregularized MLE model
    mle_model = LogisticRegression(penalty=None, fit_intercept=False, solver="lbfgs")
    mle_model.fit(X, y)
    w1_mle = mle_model.coef_[0][0]

    # TODO Step 3: Fit MAP model
    map_model = LogisticRegression(
        penalty="l2", C=C_map, fit_intercept=False, solver="lbfgs"
    )
    map_model.fit(X, y)
    w1_map = map_model.coef_[0][0]

    return w1_mle, w1_map


In [ ]:
def plot_convergence(sample_sizes: list[int] = SAMPLE_SIZES, 
                     w1_mle_list: list[float] = None, 
                     w1_map_list: list[float] = None, 
                     w1_true: float = None):
    """
    Plots estimated w1 parameters vs N on a semilog-x scale.
    """
    plt.figure(figsize=(10, 6))

    # TODO Step 1: Plot w1_mle vs N on semilog-x scale (plt.semilogx)
    plt.semilogx(sample_sizes, w1_mle_list, marker="o", label="MLE (unregularized)")
    # TODO Step 2: Plot w1_map vs N on semilog-x scale (plt.semilogx)
    plt.semilogx(sample_sizes, w1_map_list, marker="s", label="MAP (L2)")
    # TODO Step 3: Add horizontal reference line at y = w1_true (plt.axhline)
    plt.axhline(w1_true, color="red", linestyle="--", label=f"True w1 = {w1_true}")
    # TODO Step 4: Add axis labels, plot title, grid lines, and a legend
    plt.xlabel("Sample size N")
    plt.ylabel("Estimated w1")
    plt.title("Convergence of w1 estimates (MLE vs L2-MAP)")
    plt.grid(True, which="both", alpha=0.3)
    plt.legend()

    plt.show()


In [ ]:
# Initialize containers to store parameter estimates across sample sizes
w1_mle_list = []
w1_map_list = []

# TODO Step 1: Loop over each N in SAMPLE_SIZES
for N in SAMPLE_SIZES:
    X, y = generate_synthetic_data(N=N, w_true=TRUE_WEIGHTS)
    w1_mle, w1_map = fit_models(X, y, C_map=1.0)
    w1_mle_list.append(w1_mle)
    w1_map_list.append(w1_map)

# TODO Step 2: Call plot_convergence() with sample_sizes, w1_mle_list, w1_map_list, and TRUE_WEIGHTS[0]
plot_convergence(SAMPLE_SIZES, w1_mle_list, w1_map_list, TRUE_WEIGHTS[0])


**Q2(d) observation:** Both MLE and MAP estimates move toward the true w1 = 1.5 as N increases. MLE fluctuates more at small sample sizes. L2-MAP is shrunk toward zero for small N, so it is smoother and slightly biased until N is large; regularization matters less as the sample grows.


---

# Q3. Classification & Diagnostic Analysis with Logistic Regression

**Target dataset:** Scikit-learn Breast Cancer Diagnostic Dataset



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    accuracy_score
)

In [ ]:
# ============================================================
# 1. Load the Breast Cancer Diagnostic Dataset
# ============================================================

def load_breast_cancer_data():
    """
    Load sklearn's Breast Cancer Diagnostic Dataset.
    """
    data = load_breast_cancer()
    X = pd.DataFrame(data.data, columns=data.feature_names)
    y = pd.Series(data.target, name="target")
    feature_names = list(data.feature_names)
    target_names = list(data.target_names)
    return X, y, feature_names, target_names


# TODO:
X, y, feature_names, target_names = load_breast_cancer_data()
print("X shape:", X.shape)
print("y shape:", y.shape)
print("feature names:", feature_names[:5], "...")
print("target names:", target_names)
display(X.head())


## 1. Exploratory Data Analysis & Preprocessing

### 1.1 Class Distribution Check — 2.0 points

**Task:** Compute raw counts and percentages for each target class.

In [ ]:
# TODO: Implement a function that computes class counts and percentages.

def class_distribution(y):
    """
    Compute raw counts and percentages for each target class.
    """
    name_map = {0: "Malignant", 1: "Benign"}
    counts = y.value_counts().sort_index()
    distribution = pd.DataFrame(
        {
            "class_label": counts.index,
            "class_name": [name_map[i] for i in counts.index],
            "count": counts.values,
            "percentage": (counts.values / len(y) * 100).round(2),
        }
    )
    return distribution


# TODO:
distribution = class_distribution(y)
display(distribution)


**Class balance (from the table above):** Malignant (0): 212 (37.26%), Benign (1): 357 (62.74%). The dataset is imbalanced toward Benign, so accuracy alone can look high while still missing malignant cases; we therefore track malignant recall and tune the threshold.


### 1.2 Multicollinearity Analysis — 3.0 points

**Task:** Analyze the first 10 numerical features.

**Important:** The question asks for feature pairs with high collinearity. Do not merely identify the largest diagonal values (`r = 1`); exclude self-correlations.


In [ ]:
# TODO: Select the first 10 numerical features.

def get_first_n_features(X, n=10):
    """
    Return the first n numerical features from X.
    """
    selected = X.iloc[:, :n]
    return selected


# TODO:
X_first10 = get_first_n_features(X, n=10)
corr_matrix = X_first10.corr()
display(corr_matrix)


In [ ]:
# TODO: Create the correlation heatmap.

def plot_correlation_heatmap(corr_matrix):
    """
    Plot a heatmap of a correlation matrix.
    """
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        corr_matrix,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        square=True,
        linewidths=0.5,
    )
    plt.title("Correlation heatmap (first 10 features)")
    plt.tight_layout()
    plt.show()


# TODO:
plot_correlation_heatmap(corr_matrix)


In [ ]:
# TODO: Identify highly correlated feature pairs.

def find_high_correlation_pairs(corr_matrix, threshold=0.85):
    """
    Find unique feature pairs whose absolute correlation exceeds the threshold.
    """
    pairs = []
    cols = corr_matrix.columns
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            r = corr_matrix.iloc[i, j]
            if abs(r) > threshold:
                pairs.append({"feature_1": cols[i], "feature_2": cols[j], "correlation": r})
    return pd.DataFrame(pairs)


# TODO:
high_corr_pairs = find_high_correlation_pairs(corr_matrix)
display(high_corr_pairs)


**Multicollinearity interpretation:** Several pairs among the first ten features have |r| > 0.85. When features carry overlapping information, logistic coefficients are not stable individually: increasing one correlated predictor can be compensated by another, so a coefficient should not be read as an isolated causal effect.


### 1.3 Train-Test Split & Feature Scaling — 2 points

In [ ]:
# TODO: Implement train/test splitting and scaling.

def split_and_scale(X, y, test_size=0.20, random_state=2024514):
    """
    Split the data into training and test sets and standardize features.
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, y_train, y_test, scaler


# TODO:
X_train_scaled, X_test_scaled, y_train, y_test, scaler = split_and_scale(
    X, y, test_size=0.20, random_state=2024514
)


**Why fit on train only:** `fit_transform` on the test set would use test-row means and standard deviations. Those statistics are unknown at training time and would leak test information. We fit the scaler on `X_train` and apply the same transformation to `X_test`.


## 2. Model Training & Feature Coefficient Analysis

### 2.1 Baseline Model Training — 1 point

**Task:** Train a baseline `LogisticRegression` model using the scaled training data.

In [ ]:
# TODO: Train the baseline logistic regression model.

def train_logistic_regression(X_train, y_train, penalty="l2", C=1.0,
                              solver="lbfgs", max_iter=1000):
    """
    Train and return a LogisticRegression model.
    """
    model = LogisticRegression(
        penalty=penalty, C=C, solver=solver, max_iter=max_iter
    )
    model.fit(X_train, y_train)
    return model


# TODO:
baseline_model = train_logistic_regression(
    X_train_scaled,
    y_train,
    penalty="l2",
)


### 2.2 Odds Ratio Extraction — 6 points

**Remember:** Because the features were standardized, a one-unit change in a scaled feature corresponds to one standard deviation in the original feature.


In [ ]:
# TODO: Build the coefficient/odds-ratio DataFrame.

def coefficient_odds_ratio_table(model, feature_names):
    """
    Extract logistic-regression coefficients and odds ratios.
    """
    coefs = model.coef_[0]
    table = pd.DataFrame(
        {
            "feature": feature_names,
            "coefficient": coefs,
            "odds_ratio": np.exp(coefs),
            "abs_coefficient": np.abs(coefs),
        }
    )
    table = table.sort_values("abs_coefficient", ascending=False).reset_index(drop=True)
    return table


# TODO:
coef_table = coefficient_odds_ratio_table(
    baseline_model,
    X.columns,
)
display(coef_table)


In [ ]:
# TODO: Plot the 5 most positive and 5 most negative coefficients.

def plot_extreme_coefficients(coef_table, n=5):
    """
    Plot the n most positive and n most negative coefficients.
    """
    top_pos = coef_table.nlargest(n, "coefficient")
    top_neg = coef_table.nsmallest(n, "coefficient")
    plot_df = pd.concat([top_pos, top_neg]).drop_duplicates("feature")
    colors = ["green" if c > 0 else "red" for c in plot_df["coefficient"]]

    plt.figure(figsize=(10, 6))
    plt.barh(plot_df["feature"], plot_df["coefficient"], color=colors)
    plt.axvline(0, color="black", linewidth=0.8)
    plt.xlabel("Coefficient (standardized features)")
    plt.title("Five largest positive and five most negative coefficients")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()


# TODO:
plot_extreme_coefficients(coef_table, n=5)


In [ ]:
# TODO: Select a strongly negative coefficient and analyze its odds ratio.

def explain_negative_coefficient(coef_table):
    """
    Select or report a strongly negative coefficient for interpretation.
    """
    row = coef_table.loc[coef_table["coefficient"].idxmin()]
    print(
        f"Strongest negative feature: {row['feature']}\n"
        f"coefficient = {row['coefficient']:.4f}\n"
        f"odds ratio = {row['odds_ratio']:.4f}"
    )
    return row


# TODO:
neg_row = explain_negative_coefficient(coef_table)


**Negative coefficient (class 1 = Benign):** *worst texture* has coefficient −1.3506 and odds ratio 0.2591. A one-standard-deviation increase multiplies the odds of Benign by exp(β) ≈ 0.26. Because β is negative, Benign odds decrease and relative Malignant odds increase.

**L2 regularization:** The L2 penalty adds a term proportional to squared coefficient sizes, shrinking large weights toward zero. This stabilizes estimates (especially with correlated features) at the cost of some bias.


## 3. Diagnostics and Threshold Optimization

### 3.1 Default Threshold Evaluation (`p = 0.5`) — 3 points

Requirements:
1. Generate predicted probabilities on the test set.
2. Generate class predictions using the default probability threshold of `0.5`.
3. Display the confusion matrix.
4. Display the classification report.
5. Explain whether False Positives or False Negatives are more dangerous in a cancer-detection pipeline.

**Important label convention:** The assignment defines:
- `0 = Malignant`
- `1 = Benign`

Therefore, when discussing **Malignant recall**, explicitly account for this label encoding rather than blindly using the default positive-class metric.


In [ ]:
# TODO: Create a reusable prediction helper.

def get_predictions(model, X_test, threshold=0.50):
    """
    Generate probabilities and binary predictions.
    """
    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= threshold).astype(int)
    return probabilities, predictions


# TODO:
y_prob, y_pred_default = get_predictions(
    baseline_model,
    X_test_scaled,
    threshold=0.50,
)


In [ ]:
# TODO: Create a diagnostic evaluation helper.

def evaluate_classifier(y_true, y_pred):
    """
    Display classification diagnostics.
    """
    labels = [0, 1]
    target_names = ["Malignant (0)", "Benign (1)"]
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_df = pd.DataFrame(cm, index=target_names, columns=target_names)
    print("Confusion matrix:")
    display(cm_df)
    print("\nClassification report:")
    print(classification_report(y_true, y_pred, target_names=["Malignant", "Benign"]))


# TODO:
evaluate_classifier(y_test, y_pred_default)


**Medical error risk (0 = Malignant, 1 = Benign):** The more dangerous mistake is predicting Benign when the tumour is truly Malignant (true 0 → predicted 1). That is a malignant false negative and can delay treatment. Predicting Malignant for a benign case usually leads to extra tests rather than missed cancer.


### 3.2 Custom Decision Threshold Optimization — 5 points

**Goal:** Iterate through thresholds from `0.10` to `0.90` in increments of `0.05`.

The assignment asks for a threshold that:
- achieves at least **95% Recall for the Malignant class**, and
- among thresholds satisfying that requirement, **maximizes overall Precision**.

**Critical reasoning point:** Since malignant is encoded as class `0`, determine the correct way to calculate malignant recall from the confusion matrix or class-specific recall. Do not assume that `recall_score(y_true, y_pred)` automatically measures malignant recall.


In [ ]:
# TODO: Implement threshold evaluation.

def evaluate_thresholds(y_true, y_prob,
                         thresholds=None,
                         malignant_label=0,
                         min_malignant_recall=0.95):
    """
    Evaluate a sequence of decision thresholds.
    """
    if thresholds is None:
        thresholds = np.round(np.arange(0.10, 0.90 + 1e-9, 0.05), 2)

    rows = []
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        mal_recall = recall_score(
            y_true, y_pred, pos_label=malignant_label, zero_division=0
        )
        overall_precision = precision_score(
            y_true, y_pred, average="weighted", zero_division=0
        )
        rows.append(
            {
                "threshold": t,
                "malignant_recall": mal_recall,
                "weighted_precision": overall_precision,
            }
        )

    results = pd.DataFrame(rows)
    candidates = results[results["malignant_recall"] >= min_malignant_recall]
    if candidates.empty:
        best_row = results.iloc[0]
    else:
        max_prec = candidates["weighted_precision"].max()
        tied = candidates[candidates["weighted_precision"] == max_prec]
        best_row = tied.iloc[(tied["threshold"] - 0.5).abs().argmin()]
    return results, best_row


# TODO:
threshold_results, best_threshold = evaluate_thresholds(
    y_test,
    y_prob,
    malignant_label=0,
    min_malignant_recall=0.95,
)

display(threshold_results)
display(best_threshold.to_frame().T)


**Threshold metrics:** Malignant recall uses `recall_score(..., pos_label=0)`. Overall precision is weighted precision across both classes (`average='weighted'`). Among thresholds with malignant recall ≥ 0.95, we pick the highest weighted precision; if several tie, we choose the threshold closest to 0.50.


In [ ]:
# TODO: Evaluate the final custom threshold.

def evaluate_custom_threshold(y_true, y_prob, threshold):
    """
    Generate predictions at a chosen threshold and display the confusion matrix.
    """
    predictions = (y_prob >= threshold).astype(int)
    evaluate_classifier(y_true, predictions)
    return predictions


# TODO:
SELECTED_THRESHOLD = float(best_threshold["threshold"])
custom_predictions = evaluate_custom_threshold(
    y_test,
    y_prob,
    threshold=SELECTED_THRESHOLD,
)


**Observation:** The selected decision threshold is **0.45**, achieving a malignant-class recall of **95.35%** and a weighted precision of **98.29%**. The updated confusion matrix shows 41 malignant and 71 benign cases correctly classified, with 2 malignant cases misclassified as benign and no benign cases misclassified as malignant. The selected threshold satisfies the minimum 95% malignant recall requirement while maximizing weighted precision among the evaluated thresholds.